## GTB Bank Statement Parser

In [57]:
import pdfminer
import pdfplumber
import pymupdf
import pandas as pd
import re
import numpy as np

### PDF Parsing

In [58]:
tables = []
with pymupdf.open('Gtb_bs.pdf') as doc:
    print(len(doc))
    for page_number in range(len(doc)):
        page = doc.load_page(page_number)
        text_blocks = page.get_text('blocks')

        sorted_blocks = sorted(text_blocks, key=lambda b: b[4])

        table_data = []
        for block in sorted_blocks:
            # print(block)
            lines = block[4].split('\n')
            # print(lines)
            table_data.append(lines)

        if table_data:
            df = pd.DataFrame(table_data)
            processed_df = df
            # print(processed_df)

            if not processed_df.empty:
                # print(processed_df)
                tables.append(processed_df)

    if tables:
        concatenated_df = pd.concat(tables, ignore_index=True)
        bank_statement = concatenated_df
    else:
        bank_statement = pd.DataFrame()

37


In [59]:
cols = ['Trans. Date', 'Value. Date', 'Reference', 'Debits', 'Credits',	'Balance', 'Originating Branch', 'Remarks', 'a', 'b', 'c', 'd', 'e', 'f']
bank_statement.columns = cols
bank_statement.head(2)

,Trans. Date,Value. Date,Reference,Debits,Credits,Balance,Originating Branch,Remarks,a,b,c,d,e,f
0,12-Oct-2024,14-Oct-2024,' GWTR,"2,000.00","14,885.18",E- CHANNELS,TRANSFER BETWEEN CUSTOMERS via,"GTWORLD , Intra Transfer",REF:00373714910156942426200000202410120147,from AROWOSEGBE VICTOR IYANUOLUWA to,G.L.T INT'L CH,,NaN,NaN
1,12-Oct-2024,14-Oct-2024,' NIPG,"14,000.00",885.18,E- CHANNELS,NIBSS Instant Payment Outward,000013241012105029000107522445 TO,FBN/Arowosegbe Alice Moradeke,/26.875/REF:302322661301569424261400000202 f,rom AR,,NaN,NaN


### Cleaning 

In [60]:
# Removing non transaction related information
bs_df = bank_statement[~bank_statement['Trans. Date'].str.contains('^[A-Z]', regex=True)].reset_index(drop='index')
bs_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Trans. Date         407 non-null    object
 1   Value. Date         407 non-null    object
 2   Reference           406 non-null    object
 3   Debits              403 non-null    object
 4   Credits             400 non-null    object
 5   Balance             398 non-null    object
 6   Originating Branch  398 non-null    object
 7   Remarks             398 non-null    object
 8   a                   329 non-null    object
 9   b                   299 non-null    object
 10  c                   239 non-null    object
 11  d                   89 non-null     object
 12  e                   24 non-null     object
 13  f                   5 non-null      object
dtypes: object(14)
memory usage: 44.6+ KB


In [68]:
# Combining all remarks into one
bs_df['Combined Remarks'] = bs_df['Remarks'].astype(str) + bs_df['a'].astype(str) + bs_df['b'].astype(str) + bs_df['c'].astype(str) 
# + bs_df['d'].astype(str) + str(bs_df['e']) + bs_df['f']


In [69]:
bs_df.columns

Index(['Trans. Date', 'Value. Date', 'Reference', 'Debits', 'Credits',
       'Balance', 'Originating Branch', 'Remarks', 'a', 'b', 'c', 'd', 'e',
       'f', 'Combined Remarks'],
      dtype='object')

In [70]:
bs_df.head(2)

,Trans. Date,Value. Date,Reference,Debits,Credits,Balance,Originating Branch,Remarks,a,b,c,d,e,f,Combined Remarks
0,12-Oct-2024,14-Oct-2024,' GWTR,"2,000.00","14,885.18",E- CHANNELS,TRANSFER BETWEEN CUSTOMERS via,"GTWORLD , Intra Transfer",REF:00373714910156942426200000202410120147,from AROWOSEGBE VICTOR IYANUOLUWA to,G.L.T INT'L CH,,NaN,NaN,"GTWORLD , Intra TransferREF:003737149101569424..."
1,12-Oct-2024,14-Oct-2024,' NIPG,"14,000.00",885.18,E- CHANNELS,NIBSS Instant Payment Outward,000013241012105029000107522445 TO,FBN/Arowosegbe Alice Moradeke,/26.875/REF:302322661301569424261400000202 f,rom AR,,NaN,NaN,000013241012105029000107522445 TOFBN/Arowoseg...


In [74]:

bs_df['Combined Remarks'] = bs_df['Combined Remarks'].str.replace(r'[0-9]', '', regex=True)

# bs_df = bs_df.drop(columns=[ 'Remarks', 'a', 'b', 'c', 'd', 'e','f'])

bs_df.head(2)

,Trans. Date,Value. Date,Reference,Debits,Credits,Balance,Originating Branch,Combined Remarks
0,12-Oct-2024,14-Oct-2024,' GWTR,"2,000.00","14,885.18",E- CHANNELS,TRANSFER BETWEEN CUSTOMERS via,"GTWORLD , Intra TransferREF:from AROWOSEGBE V..."
1,12-Oct-2024,14-Oct-2024,' NIPG,"14,000.00",885.18,E- CHANNELS,NIBSS Instant Payment Outward,TOFBN/Arowosegbe Alice Moradeke/./REF:from AR


In [76]:
bs_df[30:50]

,Trans. Date,Value. Date,Reference,Debits,Credits,Balance,Originating Branch,Combined Remarks
30,01-Nov-2024,01-Nov-2024,'GTW,1.88,"21,447.02",635 AKIN ADESOLA,VATCHARGES,NoneNoneNone
31,01-Nov-2024,01-Nov-2024,'GTW,10.00,"27,474.65",635 AKIN ADESOLA,Commission on NIP TransferCHARGES,NoneNoneNone
32,01-Nov-2024,01-Nov-2024,'GTW,10.00,"29,485.40",635 AKIN ADESOLA,Commission on NIP TransferCHARGES,NoneNoneNone
33,01-Nov-2024,01-Nov-2024,'GTW,"2,000.00","27,484.65",635 AKIN ADESOLA,NIBSS Instant Payment Outward,NIPTRANSFER TO OPAY / TOYOSI HEPHZIBAHFALESE
34,01-Nov-2024,01-Nov-2024,'GTW,25.00,"21,448.90",635 AKIN ADESOLA,Commission on NIP TransferCHARGES,NoneNoneNone
35,01-Nov-2024,01-Nov-2024,'GTW,"3,000.00","29,495.40",635 AKIN ADESOLA,NIBSS Instant Payment Outward,NIPTRANSFER TO OPAY / PHILIP OSEMENOMOAREGBA
36,01-Nov-2024,01-Nov-2024,'GTW,"6,000.00","21,473.90",635 AKIN ADESOLA,NIBSS Instant Payment Outward,NIPTRANSFER TO OPAY / OLUMIDE HENRYADETULA
37,02-Nov-2024,02-Nov-2024,'GTW,1.88,"12,196.14",635 AKIN ADESOLA,VATCHARGES,NoneNoneNone
38,02-Nov-2024,02-Nov-2024,'GTW,25.00,"12,198.02",635 AKIN ADESOLA,Commission on NIP TransferCHARGES,NoneNoneNone
39,02-Nov-2024,02-Nov-2024,'GTW,"9,000.00","12,223.02",635 AKIN ADESOLA,NIBSS Instant Payment Outward,NIPTRANSFER TO FCMB / HABEEB ROMOKEOLADIMEJI


In [77]:
bs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Trans. Date         407 non-null    object
 1   Value. Date         407 non-null    object
 2   Reference           406 non-null    object
 3   Debits              403 non-null    object
 4   Credits             400 non-null    object
 5   Balance             398 non-null    object
 6   Originating Branch  398 non-null    object
 7   Combined Remarks    407 non-null    object
dtypes: object(8)
memory usage: 25.6+ KB


### Transformation
- Sort credits, debits and balance ambiguity
    + Ensure all credits and debits are well accounted for.
    + Ensure Balance is accurate.
- 